# Direct CNF Encoding Optimization Report

This report evaluates an optimization applied after quantum-to-CNF encoding: duplicate-clause removal, tautology removal, and subsumption. Variable identities and literal weights are preserved. A zero reduction is a valid result: it means the current encoder already avoids these redundancies.

In [ ]:
from pathlib import Path
import csv
from collections import defaultdict
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'benchmarks').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'benchmarks' / 'results' / 'cnf_encoding_optimization.csv'
if not RESULTS.exists():
    raise FileNotFoundError(f'Run benchmarks/cnf_encoding_benchmark.py first: {RESULTS}')

with RESULTS.open(encoding='utf-8', newline='') as stream:
    rows = list(csv.DictReader(stream))
rows = [row for row in rows if row['status'] == 'ok']
len(rows), RESULTS

In [ ]:
def number(row, key):
    return float(row[key])

for basis in sorted({row['basis'] for row in rows}):
    subset = [row for row in rows if row['basis'] == basis]
    print({
        'basis': basis,
        'circuits': len(subset),
        'mean_clause_reduction': sum(number(row, 'clause_reduction') for row in subset) / len(subset),
        'mean_literal_reduction': sum(number(row, 'literal_reduction') for row in subset) / len(subset),
        'duplicate_removed': sum(int(row['duplicate_removed']) for row in subset),
        'tautology_removed': sum(int(row['tautology_removed']) for row in subset),
        'subsumed_removed': sum(int(row['subsumed_removed']) for row in subset),
    })

In [ ]:
metrics = [
    ('clauses', 'optimized_clauses', 'CNF clauses'),
    ('literals', 'optimized_literals', 'Clause literals'),
    ('avg_clause_width', 'optimized_avg_clause_width', 'Average clause width'),
    ('weighted_variables', 'optimized_weighted_variables', 'Weighted variables'),
]

figure, axes = plt.subplots(2, 2, figsize=(13, 9))
for axis, (before_key, after_key, title) in zip(axes.flat, metrics):
    for basis, color in [('computational', '#2a6fbb'), ('pauli', '#e07a2d')]:
        subset = [row for row in rows if row['basis'] == basis]
        x = list(range(len(subset)))
        axis.plot(x, [number(row, before_key) for row in subset], color=color, alpha=0.35, label=f'{basis} baseline')
        axis.plot(x, [number(row, after_key) for row in subset], color=color, label=f'{basis} optimized')
    axis.set_title(title)
    axis.grid(axis='y', alpha=0.25)
    axis.legend(fontsize=8)
figure.suptitle('Direct CNF encoding: baseline versus clause optimization')
figure.tight_layout()
plt.show()

In [ ]:
removal_types = ['duplicate_removed', 'tautology_removed', 'subsumed_removed']
totals = [sum(int(row[key]) for row in rows) for key in removal_types]
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(['duplicate', 'tautology', 'subsumption'], totals, color=['#4c78a8', '#f58518', '#54a24b'])
axis.set_ylabel('Removed clauses')
axis.set_title('Why clauses were removed')
axis.grid(axis='y', alpha=0.25)
plt.show()

## Conclusion

If all three removal counts are zero, this optimization is not the right bottleneck for the current generated encodings. The next encoding-level candidates are gate-template replacement, Clifford/GF(2) block fusion, weighted sign-variable compression, or BDD-assisted generation for the large controlled-gate sign functions.